# 03.1 Simple Chromatography

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Binding-model attributes, event calls, parameter changes and simulation.

**Introduces:** `StericMassAction`, salt-dependent binding and a complete load–wash–elute separation. Pore transport is an additional topic.

1. The separation goal
2. Assemble the process
3. Salt-dependent binding
4. Run and compare gradients
5. Beyond lumped transport (additional)

In [ ]:
%matplotlib inline
# Load the familiar column setup and the supplied plotting functions.
%run resources/chromatography_setup.py
from CADETProcess.simulator import Cadet

## The separation goal

A mixture contains **Protein A** and **Protein B**. We want to collect Protein A separately: load the mixture onto an ion-exchange column, wash and release the proteins with a salt gradient.

```{figure} resources/lwe_separation.png
:width: 85%

Simulated example: the inlet method produces two protein peaks at the outlet. Salt uses a separate scale.
```

The question is now **which method separates the proteins**, rather than just when a tracer arrives. We will build this example and compare it with a shorter gradient.

## Assemble the process

```{figure} resources/separation_system.svg
:width: 50%
```

We reuse **Inlet → Column → Outlet** and the lumped column model from First Model: **10 × 2 cm**, porosity **0.4**, dispersion **1e-7 m²/s**, flow **6 mL/min**. Here, “without pores” means the model lumps the liquid inside and between particles; it does not require a nonporous resin.

| Familiar piece | Its role in this process |
|---|---|
| Transport and RTD | Describe delay and spreading; transport settings remain fixed |
| Binding | Retain proteins on the resin |
| Inlet events | Define load, wash, elution and final wash |
| `Cadet` | Simulate the connected process |

This example represents the column alone. A lab detector can also include instrument delay and spreading, as in RTD; a measured chromatogram needs that system contribution accounted for.

The [setup script](resources/chromatography_setup.py) contains the familiar component, unit and flow-sheet construction. The next cell loads a fresh column with **Salt, Protein A, Protein B**, initially equilibrated with **50 mM salt** and no protein.

In [ ]:
# Load the familiar column and flow sheet; add binding and events in the following steps.
column, process = separation_model("lwe", with_binding=False)
process.cycle_time = 52 * 60  # 52 min, converted to seconds

## Salt-dependent binding

An inlet gradient alone does not make a protein elute: the binding model must respond to salt. In ion exchange, increasing salt weakens protein retention. Different binding strengths make the two proteins elute at different times.

The Steric Mass Action model represents this competition ([equations in CADET-Core](https://cadet.github.io/master/modelling/binding/steric_mass_action.html), [`StericMassAction` in CADET-Process](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.StericMassAction.html)). **Salt is the first component**, followed by Protein A and Protein B in every component-wise parameter list.

| Attribute | Meaning |
|---|---|
| `adsorption_rate`, `desorption_rate` | Binding and release rates |
| `characteristic_charge` | Number of resin sites involved in binding a protein |
| `steric_factor` | Additional sites shielded by a bound protein |
| `capacity` | Total ionic capacity of the resin |

```{note}
Each list has one entry per component: **[Salt, Protein A, Protein B]**. Salt's entries in `adsorption_rate`, `desorption_rate` and `steric_factor` are ignored; we use **0** as placeholders. Salt still controls protein release through its concentration.

The first `characteristic_charge` entry is the **salt valence**: use **1** for monovalent salt. `capacity` and the reference concentrations are single values for the whole model. A list with the wrong length raises `ValueError: Expected size …` ([99.6 Common Problems](../../99%20Misc/99.6%20Common%20Problems/99.6a_common_problems_lesson.ipynb)).
```

In [ ]:
# SMA binding model
from CADETProcess.processModel import StericMassAction

binding = StericMassAction(process.component_system, "SMA")
binding.is_kinetic = True
binding.adsorption_rate = [0.0, 0.3, 0.2]  # Salt placeholder, Protein A, Protein B
binding.desorption_rate = [0.0, 1.5, 1.5]
binding.characteristic_charge = [1, 7, 5]  # Salt valence, then protein charges
binding.steric_factor = [0, 50, 50]
binding.capacity = 225
binding.reference_liquid_phase_conc = 300
binding.reference_solid_phase_conc = 225
column.binding_model = binding
column.q = [225, 0, 0]

`column.q` initializes unloaded resin: its bound counter-ion state equals the ionic capacity, and both protein loadings are zero. These illustrative SMA parameters are supplied; the Linear fit from Binding cannot determine them. Later parameter estimation uses experimental data to fit a chosen binding model.

```{dropdown} The equation and reference concentrations
For a protein, the normalized SMA rate equation is

$$
\frac{dq_i}{dt} = k_{a,i}c_i\left(\frac{\bar q_0}{q_{ref}}\right)^{\nu_i}

- k_{d,i}q_i\left(\frac{c_{salt}}{c_{ref}}\right)^{\nu_i},
\qquad
\bar q_0=\Lambda-\sum_j(\nu_j+\sigma_j)q_j.
$$

The liquid state $c_i$ and bound state $q_i$ are familiar from Binding. Increasing salt strengthens the release term. The reference concentrations normalize the rate expression; our rates are defined with `reference_liquid_phase_conc=300` and `reference_solid_phase_conc=225`. These values belong to the rate parameters and must be kept with them when the model is reused.
```

## Run the method

The method is a load–wash–elute sequence: both proteins are loaded in low salt, unbound material is washed out, and a rising salt gradient releases the proteins again. Concentrations in mM are numerically equal to mol/m³.

| Event | Starts | Salt | Protein A, B | Duration |
|---|---:|---|---:|---:|
| `load` | 0 min | 50 mM | 0.1 mM | 2 min |
| `wash` | 2 min | 50 mM | 0 | 10 min |
| `elute` | 12 min | 50 → 500 mM | 0 | 30 min |
| `final_wash` | 42 min | 500 mM | 0 | 10 min |

The method takes **52 min**. This column holds **31.4 mL** per column volume (CV); at **6 mL/min**, one CV takes **5.24 min**, so the wash corresponds to about **1.9 CV** and the gradient to **5.7 CV**. The short load keeps the column below overload: the example shows separation, not maximum resin utilization.

Each row becomes one `add_event()`, as in Inlet Profiles; now the events feed a column with salt-dependent binding. Event times are in seconds, and the gradient is a slope: 450 mM over 1800 s gives **0.25 mM/s**.

In [ ]:
# Method events: load, wash, elute and final wash (times in seconds).
salt_slope = (500 - 50) / (30 * 60)
process.add_event("load", "flow_sheet.inlet.c", [50, 0.1, 0.1], 0)
process.add_event("wash", "flow_sheet.inlet.c", [50, 0, 0], 2 * 60)
process.add_event("elute", "flow_sheet.inlet.c", [[50, salt_slope], [0], [0]], 12 * 60)
process.add_event("final_wash", "flow_sheet.inlet.c", [500, 0, 0], 42 * 60)

In [ ]:
# Simulate the method and compare its inlet profile with the outlet chromatogram.
simulator = Cadet()
baseline_results = simulator.simulate(process)
_ = plot_lwe(baseline_results)

The feed enters together, but the proteins leave at different times. **Protein B elutes first; Protein A elutes later.** The inlet salt gradient reaches the column outlet after a transport delay. Retention and transport together determine the chromatogram.

## Compare gradients

What happens when the gradient is shortened from **30 to 15 min**? Load, wash and the salt endpoints stay the same. The salt rises over half the time, so the slope doubles, and the final wash and the end of the method move 15 min earlier:

| Setting | 30 min gradient | 15 min gradient |
|---|---:|---:|
| Salt slope | 450 mM / 1800 s = 0.25 mM/s | 450 mM / 900 s = 0.5 mM/s |
| Final wash starts | 42 min | 27 min |
| Cycle time | 52 min | 37 min |

The existing events are changed in place: `process.elute.state` takes the new slope, `process.final_wash.time` the new start time.

In [ ]:
# Shorten the gradient: change the elution event and move the final wash and the cycle end.
salt_slope = (500 - 50) / (15 * 60)
process.elute.state = [[50, salt_slope], [0], [0]]
process.final_wash.time = 27 * 60
process.cycle_time = 37 * 60

In [ ]:
# Keep the original results and compare them with the newly simulated method.
short_results = simulator.simulate(process)
_ = plot_gradient_comparison(
    baseline_results, short_results, "30 min gradient", "15 min gradient"
)

The shorter gradient finishes sooner and brings the peaks closer together. Choosing a method involves both **separation and run time**. Peak spacing alone does not establish product purity; defining collection windows and evaluating purity and recovery follows in Fractionation.

## Beyond lumped transport

The main example keeps one liquid state so we can concentrate on process setup. When particle transport affects peak width, a model with pores can describe it explicitly.

| Model | Additional transport detail |
|---|---|
| `LumpedRateModelWithPores` | Separate bulk and pore liquid; transfer across the particle surface |
| `GeneralRateModel` | Concentration gradients within particles, including pore diffusion |

```{dropdown} Bulk, pore and bound states
For a column model with pores:

| State | Equation describes | Recorded solution |
|---|---|---|
| Bulk liquid $c$ | Convection, dispersion and transfer into particles | `results.solution.column.bulk` |
| Pore liquid $c_p$ | Surface transfer; pore diffusion in `GeneralRateModel` | `results.solution.column.particle` |
| Bound $q$ | Binding, driven by the pore concentration | `results.solution.column.solid` |

The column's `solution_recorder.write_solution_bulk`, `write_solution_particle` and `write_solution_solid` flags save these internal states. The detector measures the outlet concentration. The current `LumpedRateModelWithoutPores` has no separate pore-liquid state.
```

## Summary

- Column transport, salt-dependent binding and inlet events define the separation.
- Changing gradient duration changes run time and peak spacing.
- Product purity requires collection windows; peak spacing alone is insufficient.

**Now:** [**03.1 Simple Chromatography: Exercises**](03.1b_simple_chromatography_exercises.ipynb): complete a load–wash–elute method and compare a shorter gradient.

**Next lesson:** [**03.2 Advanced Chromatography**](../03.2%20Advanced%20Chromatography/03.2a_advanced_chromatography_lesson.ipynb) adds instrument transport and multiple columns.